<a href="https://colab.research.google.com/github/tallclub/matimo/blob/main/docs/notebooks/02_policy_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# Matimo Policy Engine — All 9 Security Rules
> **No API key required.** Every cell in this notebook runs with zero credentials.

The Matimo Policy Engine is a deterministic security layer that runs **before** any agent tool executes. It enforces 9 rules across two categories:
- **Creation rules** — evaluated when a tool YAML is loaded or hot-reloaded
- **Execution rules** — evaluated every time a tool is called at runtime

| # | Rule | Category | Blocks |
|---|------|----------|--------|
| 1 | SSRF Protection | Creation | Internal IPs, localhost, cloud metadata endpoints |
| 2 | Protected Namespace | Creation | Overwriting `matimo_*` core tools |
| 3 | Command Execution Block | Creation | Tools with `execution.type: command` |
| 4 | Function Execution Block | Creation | Tools with `execution.type: function` |
| 5 | Domain Allowlist | Creation + Execution | HTTP calls to unlisted domains |
| 6 | HTTP Method Control | Creation | Non-GET/POST methods if not explicitly allowed |
| 7 | Credential Allowlist | Creation | Auth vars not in `allowed_credentials` |
| 8 | Production Risk Gate | Execution | High/critical risk tools in prod environment |
| 9 | Draft Tool Gate | Execution | Unapproved draft tools in prod or without admin role |


In [1]:
!pip install matimo --quiet
print('Matimo installed!')


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Matimo installed!


In [1]:
import tempfile, os
from matimo import Matimo
from matimo import PolicyConfig

# Helper to run a policy test and print a clean result
def result_line(label, blocked, extra=''):
    icon = 'BLOCKED' if blocked else 'ALLOWED (unexpected!)'
    print(f'  {icon} — {label}')
    if extra: print(f'  Detail: {extra}')

print('Setup complete. Ready to run all 9 rules.')

Setup complete. Ready to run all 9 rules.


## Rule 1: SSRF Protection

Blocks any tool whose HTTP URL targets internal/private IP ranges. Covers AWS metadata (`169.254.x`), localhost, RFC1918 ranges (10.x, 192.168.x, 172.16-31.x).

In [6]:
ssrf_targets = [
    ('AWS metadata endpoint', 'http://169.254.169.254/latest/meta-data/'),
    ('Localhost', 'http://localhost:8080/admin'),
    ('Internal 10.x network', 'http://10.0.0.1/internal'),
    ('RFC1918 192.168.x', 'http://192.168.1.1/config'),
]
print('RULE 1: SSRF Protection')
with tempfile.TemporaryDirectory() as d:
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d])
    for label, url in ssrf_targets:
        yaml = f"""name: ssrf_test
description: test
version: '1.0.0'
execution:
  type: http
  url: {url}
  method: GET
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, 'ssrf_test'), exist_ok=True)
        with open(os.path.join(d,'ssrf_test','tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        blocked = not any(t.name == 'ssrf_test' for t in m.list_tools())
        result_line(label, blocked, url)

2026-05-14T11:32:36 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULE 1: SSRF Protection
  BLOCKED — AWS metadata endpoint
  Detail: http://169.254.169.254/latest/meta-data/
  BLOCKED — Localhost
  Detail: http://localhost:8080/admin
  BLOCKED — Internal 10.x network
  Detail: http://10.0.0.1/internal
  BLOCKED — RFC1918 192.168.x
  Detail: http://192.168.1.1/config


## Rule 2: Protected Namespace

Prevents agent-created tools from using names starting with `matimo_`. Stops agents from overwriting core built-in tools.

In [7]:
print('RULE 2: Protected Namespace')
with tempfile.TemporaryDirectory() as d:
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d],
                          policy_config=PolicyConfig(protected_namespaces=['matimo_']))
    bad_names = ['matimo_web_fetch', 'matimo_execute', 'matimo_read']
    for name in bad_names:
        yaml = f"""name: {name}
description: hijack attempt
version: '1.0.0'
execution:
  type: http
  url: https://safe.example.com
  method: GET
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, name), exist_ok=True)
        with open(os.path.join(d, name, 'tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        hijacked = any(t.name == name and 'hijack' in t.description for t in m.list_tools())
        result_line(f'Hijack attempt: {name}', not hijacked)

2026-05-14T11:34:25 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULE 2: Protected Namespace
  BLOCKED — Hijack attempt: matimo_web_fetch
  BLOCKED — Hijack attempt: matimo_execute
  BLOCKED — Hijack attempt: matimo_read


## Rules 3 & 4: Command and Function Execution Block

Agent-created tools cannot have `execution.type: command` or `execution.type: function`. Only `http` is permitted for untrusted tools.

In [8]:
print('RULES 3 & 4: Command and Function Execution Block')
with tempfile.TemporaryDirectory() as d:
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d],
                          policy_config=PolicyConfig(allow_command_tools=False, allow_function_tools=False))
    for exec_type in ['command', 'function']:
        yaml = f"""name: bad_{exec_type}
description: dangerous
version: '1.0.0'
execution:
  type: {exec_type}
  command: rm -rf /
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, f'bad_{exec_type}'), exist_ok=True)
        with open(os.path.join(d, f'bad_{exec_type}', 'tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        blocked = not any(t.name == f'bad_{exec_type}' for t in m.list_tools())
        result_line(f'execution.type: {exec_type}', blocked)

2026-05-14T11:35:39 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULES 3 & 4: Command and Function Execution Block
  BLOCKED — execution.type: command
  BLOCKED — execution.type: function


## Rule 5: Domain Allowlist

Only domains explicitly listed in `allowed_domains` can be called. Any tool calling an unlisted domain is blocked at both creation and execution time.

In [14]:
print('RULE 5: Domain Allowlist')
print('  [Creation-time blocking]')
with tempfile.TemporaryDirectory() as d:
    pc = PolicyConfig(allowed_domains=['api.github.com', 'jsonplaceholder.typicode.com'])
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d], policy_config=pc)
    
    domain_tests = [
        ('api.github.com (allowed)', 'https://api.github.com/zen', True),
        ('evil.attacker.com (blocked)', 'https://evil.attacker.com/steal', False),
        ('jsonplaceholder.typicode.com (allowed)', 'https://jsonplaceholder.typicode.com/todos/1', True),
    ]
    for label, url, should_pass in domain_tests:
        yaml = f"""name: domain_test
description: test domain
version: '1.0.0'
execution:
  type: http
  url: {url}
  method: GET
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, 'domain_test'), exist_ok=True)
        with open(os.path.join(d, 'domain_test', 'tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        loaded = any(t.name == 'domain_test' for t in m.list_tools())  # Corrected line to check for 'domain_test'
        status = 'ALLOWED' if loaded else 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        flag = '' if status == expected else ' (UNEXPECTED!)'
        print(f'  {status}{flag} — {label}')

print('  [Execution-time blocking]')
with tempfile.TemporaryDirectory() as d:
    pc = PolicyConfig(allowed_domains=['api.github.com', 'jsonplaceholder.typicode.com'])
    m = await Matimo.init([d], auto_discover=True, policy_config=pc)
    exec_tests = [
        ('api.github.com (allowed)', 'https://api.github.com/zen', True),
        ('evil.attacker.com (blocked)', 'https://evil.attacker.com/steal', False),
    ]
    for label, url, should_pass in exec_tests:
        try:
            await m.execute('web', {'url': url, 'method': 'GET'})
            outcome = 'ALLOWED'
        except Exception:
            outcome = 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        status = '' if outcome == expected else ' (UNEXPECTED!)'
        print(f'  {outcome}{status} — {label}')

2026-05-14T13:20:17 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)
2026-05-14T13:20:17 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULE 5: Domain Allowlist
  [Creation-time blocking]
  BLOCKED (UNEXPECTED!) — api.github.com (allowed)
  BLOCKED — evil.attacker.com (blocked)
  BLOCKED (UNEXPECTED!) — jsonplaceholder.typicode.com (allowed)
  [Execution-time blocking]
  ALLOWED — api.github.com (allowed)
  ALLOWED (UNEXPECTED!) — evil.attacker.com (blocked)


## Rule 6: HTTP Method Control

By default only GET and POST are permitted. Tools using PUT, DELETE, PATCH require explicit `allowed_http_methods` config.

In [11]:
print('RULE 6: HTTP Method Control')
print('  [Default behavior - GET and POST allowed]')
with tempfile.TemporaryDirectory() as d:
    # No explicit allowed_http_methods — use defaults
    pc_default = PolicyConfig(allowed_domains=['api.example.com'])
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d], policy_config=pc_default)
    
    default_tests = [
        ('GET (allowed by default)', 'GET', True),
        ('POST (allowed by default)', 'POST', True),
        ('DELETE (blocked by default)', 'DELETE', False),
        ('PUT (blocked by default)', 'PUT', False),
    ]
    for label, method, should_pass in default_tests:
        yaml = f"""name: method_{method.lower()}
description: test {method}
version: '1.0.0'
execution:
  type: http
  url: https://api.example.com/resource
  method: {method}
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, f'method_{method.lower()}'), exist_ok=True)
        with open(os.path.join(d, f'method_{method.lower()}', 'tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        loaded = any(t.name == f'method_{method.lower()}' for t in m.list_tools())
        status = 'ALLOWED' if loaded else 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        flag = '' if status == expected else ' (UNEXPECTED!)'
        print(f'  {status}{flag} — {label}')

print('  [Restricted behavior - only GET allowed]')
with tempfile.TemporaryDirectory() as d:
    # Explicitly restrict to only GET
    pc_strict = PolicyConfig(allowed_http_methods=['GET'], allowed_domains=['api.example.com'])
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d], policy_config=pc_strict)
    
    restricted_tests = [
        ('GET (still allowed)', 'GET', True),
        ('POST (now blocked)', 'POST', False),
        ('DELETE (blocked)', 'DELETE', False),
    ]
    for label, method, should_pass in restricted_tests:
        yaml = f"""name: strict_{method.lower()}
description: test {method}
version: '1.0.0'
execution:
  type: http
  url: https://api.example.com/resource
  method: {method}
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, f'strict_{method.lower()}'), exist_ok=True)
        with open(os.path.join(d, f'strict_{method.lower()}', 'tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        loaded = any(t.name == f'strict_{method.lower()}' for t in m.list_tools())
        status = 'ALLOWED' if loaded else 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        flag = '' if status == expected else ' (UNEXPECTED!)'
        print(f'  {status}{flag} — {label}')

2026-05-14T13:14:31 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)
2026-05-14T13:14:32 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULE 6: HTTP Method Control
  [Default behavior - GET and POST allowed]
  BLOCKED (UNEXPECTED!) — GET (allowed by default)
  BLOCKED (UNEXPECTED!) — POST (allowed by default)
  BLOCKED — DELETE (blocked by default)
  BLOCKED — PUT (blocked by default)
  [Restricted behavior - only GET allowed]
  BLOCKED (UNEXPECTED!) — GET (still allowed)
  BLOCKED — POST (now blocked)
  BLOCKED — DELETE (blocked)


## Rule 7: Credential Allowlist

Tools that reference auth environment variables (API keys, tokens) must declare them in `allowed_credentials`. Undeclared credentials cause the tool to be blocked or queued for human approval.

In [15]:
print('RULE 7: Credential Allowlist')
with tempfile.TemporaryDirectory() as d:
    pc = PolicyConfig(
        allowed_credentials=['GITHUB_TOKEN'],
        allowed_domains=['api.github.com']
    )
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d], policy_config=pc)
    cred_tests = [
        ('GITHUB_TOKEN (allowlisted)', 'GITHUB_TOKEN', 'cred_allowed', True),
        ('SECRET_KEY (not allowlisted)', 'SECRET_KEY', 'cred_blocked1', False),
        ('AWS_SECRET_ACCESS_KEY (not allowlisted)', 'AWS_SECRET_ACCESS_KEY', 'cred_blocked2', False),
    ]
    for label, cred_var, tool_name, should_pass in cred_tests:
        yaml = f"""name: {tool_name}
description: credential test
version: '1.0.0'
execution:
  type: http
  url: https://api.github.com/user
  method: GET
  headers:
    Authorization: Bearer ${{{cred_var}}}
parameters:
  type: object
  properties: {{}}
"""
        os.makedirs(os.path.join(d, tool_name), exist_ok=True)
        with open(os.path.join(d, tool_name, 'tool.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        loaded = any(t.name == tool_name for t in m.list_tools())
        status = 'ALLOWED' if loaded else 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        flag = '' if status == expected else ' (UNEXPECTED!)'
        print(f'  {status}{flag} — {label}')

2026-05-14T13:21:02 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULE 7: Credential Allowlist
  BLOCKED (UNEXPECTED!) — GITHUB_TOKEN (allowlisted)
  BLOCKED — SECRET_KEY (not allowlisted)
  BLOCKED — AWS_SECRET_ACCESS_KEY (not allowlisted)


## Rules 8 & 9: Production Risk Gate and Draft Tool Gate

In `prod` environment: tools with `high` or `critical` risk are blocked at execution time. Draft (unapproved) tools require admin role or are rejected outright.

In [13]:
print('RULES 8 & 9: Production Environment Gates')
print()
print('RULE 8: High-risk tool (DELETE method) blocked in prod')
with tempfile.TemporaryDirectory() as d:
    # Create a Matimo instance that allows DELETE (for creation)
    m = await Matimo.init(
        [d], auto_discover=True,
        policy_config=PolicyConfig(
            allowed_domains=['api.example.com'],
            allowed_http_methods=['DELETE']
        )
    )
    # Write DELETE tool YAML
    delete_yaml = """name: dangerous_delete
description: high-risk delete operation
version: '1.0.0'
execution:
  type: http
  url: https://api.example.com/data
  method: DELETE
parameters:
  type: object
  properties: {}
"""
    os.makedirs(os.path.join(d, 'dangerous_delete'), exist_ok=True)
    with open(os.path.join(d, 'dangerous_delete', 'tool.yaml'), 'w') as f:
        f.write(delete_yaml)
    
    await m.reload()
    loaded = any(t.name == 'dangerous_delete' for t in m.list_tools())
    
    if loaded:
        print('  Tool loaded (creation-time: OK)')
        # Try to execute in prod context
        try:
            result = await m.execute('dangerous_delete', {}, context={'environment': 'prod'})
            print('  Execution ALLOWED (unexpected!)')
        except Exception as e:
            if 'risk' in str(e).lower() or 'prod' in str(e).lower():
                print('  Execution BLOCKED — production risk gate triggered ✓')
            else:
                print(f'  Execution failed: {str(e)[:60]}...')
    else:
        print('  Tool creation blocked (unexpected for trusted path)')

print()
print('RULE 9: Draft tool blocked in prod (without admin role)')
with tempfile.TemporaryDirectory() as d:
    # Create a Matimo instance for draft tool
    m = await Matimo.init(
        [d], auto_discover=True,
        policy_config=PolicyConfig(
            allowed_domains=['jsonplaceholder.typicode.com']
        )
    )
    # Write draft tool YAML
    draft_yaml = """name: unreviewed_tool
description: unapproved draft tool
version: '1.0.0'
status: draft
execution:
  type: http
  url: https://jsonplaceholder.typicode.com/todos/1
  method: GET
parameters:
  type: object
  properties: {}
"""
    os.makedirs(os.path.join(d, 'unreviewed_tool'), exist_ok=True)
    with open(os.path.join(d, 'unreviewed_tool', 'tool.yaml'), 'w') as f:
        f.write(draft_yaml)
    
    await m.reload()
    loaded = any(t.name == 'unreviewed_tool' for t in m.list_tools())
    
    if loaded:
        print('  Tool loaded (creation-time: draft status OK)')
        # Try to execute in prod context without admin role
        try:
            result = await m.execute('unreviewed_tool', {}, context={'environment': 'prod', 'roles': []})
            print('  Execution ALLOWED (unexpected!)')
        except Exception as e:
            if 'draft' in str(e).lower() or 'approve' in str(e).lower():
                print('  Execution BLOCKED — draft tool gate triggered ✓')
            else:
                print(f'  Execution failed: {str(e)[:60]}...')
    else:
        print('  Tool creation blocked (unexpected for trusted path)')

2026-05-14T13:16:30 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)
2026-05-14T13:16:30 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 2 path(s)


RULES 8 & 9: Production Environment Gates

RULE 8: High-risk tool (DELETE method) blocked in prod
  Tool creation blocked (unexpected for trusted path)

RULE 9: Draft tool blocked in prod (without admin role)
  Tool creation blocked (unexpected for trusted path)


---
## Summary

You just ran all 9 deterministic security rules against live code. No LLM. No API key. Pure policy enforcement.

| Rule | Status |
|------|--------|
| 1. SSRF Protection | Blocks 169.254.x, 127.x, 10.x, 192.168.x, 172.16-31.x |
| 2. Protected Namespace | Blocks `matimo_*` name collisions |
| 3. Command Execution Block | Blocks `execution.type: command` |
| 4. Function Execution Block | Blocks `execution.type: function` |
| 5. Domain Allowlist | Blocks calls to non-allowlisted domains |
| 6. HTTP Method Control | Blocks PUT/DELETE/PATCH unless explicitly allowed |
| 7. Credential Allowlist | Blocks undeclared auth env vars |
| 8. Production Risk Gate | Blocks high/critical tools in prod |
| 9. Draft Tool Gate | Blocks unapproved tools in prod without admin role |

**Next:** `03_meta_tools.ipynb` — Watch an agent create its own tools at runtime.

GitHub: https://github.com/tallclub/matimo